# Test gender assignment accuracy

I have a test file, and I want to check what the transformer based assignments are and what the regex assignments might be.

In [1]:
from pathlib import Path
import pandas as pd
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True, nb_workers=12)

root_dir = Path.cwd().parent
data_dir = root_dir / "data"
results_dir = data_dir / "olmo7b_results" / "v2"
test_results = results_dir / "processed_test.json"

INFO: Pandarallel will run on 12 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.

https://nalepae.github.io/pandarallel/troubleshooting/


In [8]:
def assign_gender_spacy(text):
    import spacy
    from collections import Counter

    nlp = spacy.load("en_core_web_sm")
    if not isinstance(text, str): return None
    
    doc = nlp(text.lower())
    
    # Tally up all pronouns in the text
    pronouns = [token.text for token in doc if token.pos_ == "PRON"]
    counts = Counter(pronouns)
    
    f_score = counts['she'] + counts['her'] + counts['hers']
    m_score = counts['he'] + counts['him'] + counts['his']
    
    if f_score > m_score:
        return 'female'
    elif m_score > f_score:
        return 'male'
    return 'unspecified'

def extract_gender(text):
    import spacy
    
    nlp = spacy.load("en_core_web_sm")
    if not isinstance(text, str):
        return None
    
    doc = nlp(text.lower())
    
    female_terms = {'she','her','hers','female','woman','girl'}
    male_terms   = {'he','him','his','male','man','boy'}
    
    f_score = 0
    m_score = 0
    
    for token in doc:
        lemma = token.lemma_
        
        if lemma in female_terms:
            f_score += 1
        elif lemma in male_terms:
            m_score += 1
    
    if f_score > m_score:
        return 'female'
    elif m_score > f_score:
        return 'male'
    print(f"Text: {text}\nFemale score: {f_score}, Male score: {m_score}")
    return 'unspecified'

## Some stats for the test file

In [3]:
df = pd.read_json(test_results)
df.describe()

,profile_id,temperature,response_number,mean_entropy,max_entropy,min_entropy,std_entropy,mean_entropy_nucleus,max_entropy_nucleus,min_entropy_nucleus,std_entropy_nucleus
count,2119.000000,2119.000000,2119.000000,2119.000000,2119.000000,2.119000e+03,2119.000000,2119.000000,2119.000000,2119.0,2119.000000
mean,94.905144,0.409297,5.497876,0.366589,2.077723,1.347332e-07,0.440176,0.259897,1.766053,0.0,0.377460
std,88.204843,0.189586,2.871972,0.219103,0.706773,5.328208e-07,0.161622,0.156688,0.575357,0.0,0.130016
min,1.000000,0.200000,1.000000,0.068629,0.992188,0.000000e+00,0.182031,0.044229,0.867188,0.0,0.156374
25%,18.000000,0.200000,3.000000,0.149807,1.406250,1.047607e-26,0.275274,0.106532,1.250000,0.0,0.245750
50%,36.000000,0.500000,5.000000,0.390175,2.109375,1.000000e-10,0.477016,0.273962,1.750000,0.0,0.404810
75%,194.000000,0.500000,8.000000,0.492613,2.500000,1.100000e-09,0.536581,0.350090,2.109375,0.0,0.455745
max,220.000000,0.700000,10.000000,1.011082,4.593750,1.007320e-05,0.879247,0.726901,4.125000,0.0,0.739570


In [4]:
df[df['gender'] == 'unspecified'].head()

,model_key,model_name,profile_id,temperature,occupation_category,attended_university,response_number,response,mean_entropy,max_entropy,min_entropy,std_entropy,mean_entropy_nucleus,max_entropy_nucleus,min_entropy_nucleus,std_entropy_nucleus,gender
13,base,allenai/Olmo-3-1025-7B,2,0.2,non-commissioned officers in regular armed forces,no,4,"male, here is a personal narrative based on t...",0.165698,1.296875,1.527235e-27,0.277427,0.119225,1.093750,0,0.254834,unspecified
18,base,allenai/Olmo-3-1025-7B,2,0.2,non-commissioned officers in regular armed forces,no,9,"male, let's name him John. John is a 30-year-...",0.180097,1.328125,1.744834e-25,0.304142,0.131868,1.218750,0,0.269337,unspecified
27,base,allenai/Olmo-3-1025-7B,3,0.2,other ranks in regular armed forces,no,8,"male, let's create a narrative for a 25-year-...",0.172913,1.640625,3.655266e-26,0.310473,0.125882,1.414062,0,0.277805,unspecified
30,base,allenai/Olmo-3-1025-7B,4,0.2,"managing directors, board members, senior admi...",no,1,"male, here is a personal narrative based on t...",0.160157,1.226562,1.175403e-28,0.279393,0.114772,1.093750,0,0.254432,unspecified
34,base,allenai/Olmo-3-1025-7B,4,0.2,"managing directors, board members, senior admi...",no,5,"male, let's create a narrative for a man name...",0.190919,1.515625,1.751271e-28,0.309026,0.152204,1.414062,0,0.281329,unspecified


In [5]:
gender = df['gender'].to_list()
df['test_gender'] = df['response'].parallel_apply(assign_gender_spacy)
test_gender = df['test_gender'].to_list()

In [6]:
from collections import Counter
Counter(test_gender)

Counter({'male': 1061, 'female': 1028, 'unspecified': 30})

In [12]:
unspecified_entries = df.loc[df["test_gender"] == "unspecified", ["profile_id", "response_number", "temperature", "gender"]]
unspecified_entries

,profile_id,response_number,temperature,gender
461,179,2,0.2,unspecified
989,11,10,0.5,unspecified
1147,27,8,0.5,unspecified
1607,205,8,0.5,unspecified
1617,206,8,0.5,unspecified
1645,209,6,0.5,unspecified
1758,220,9,0.5,unspecified
1801,5,2,0.7,unspecified
1855,10,6,0.7,unspecified
1860,11,1,0.7,unspecified


In [11]:
# if test_gender is unspecified but gender is specified, then change test_gender to match gender
df.loc[(df["test_gender"] == "unspecified") & (df["gender"] != "unspecified"), "test_gender"] = df["gender"]
Counter(df["test_gender"])

Counter({'male': 1069, 'female': 1035, 'unspecified': 15})

In [14]:
# Crosstab showing counts of gender vs test_gender combinations
ct = pd.crosstab(df['gender'], df['test_gender'], margins=True, margins_name='Total')
print(ct)

# Quick match rate summary
match = (df['gender'] == df['test_gender'])
print(f"\nMatching: {match.sum()} / {len(df)} ({match.mean():.1%})")

test_gender  female  male  unspecified  Total
gender                                       
female          826     6            0    832
male             10   736            0    746
unspecified     199   327           15    541
Total          1035  1069           15   2119

Matching: 1577 / 2119 (74.4%)


# Spacy for everything

beyond pronouns

In [9]:
gender = df['gender'].to_list()
df['test_gender'] = df['response'].parallel_apply(extract_gender)
test_gender = df['test_gender'].to_list()

In [10]:
from collections import Counter
Counter(test_gender)

Counter({'male': 1076, 'female': 1041, 'unspecified': 2})

In [12]:
unspecified_entries = df.loc[df["test_gender"] == "unspecified", ["profile_id", "response_number", "temperature", "gender", "response"]]
unspecified_entries

,profile_id,response_number,temperature,gender,response
1147,27,8,0.5,unspecified,"female, the narrative begins with a brief int..."
1950,20,1,0.7,unspecified,female:\nThe name of the female individual is...


In [ ]:
for idx, row in unspecified_entries.iterrows():
    print(f"ID: {row['profile_id']}, Response #: {row['response_number']}, gender: {extract_gender(row['response'])}")

Text:  female, the narrative begins with a brief introduction of the character, highlighting her background and early life experiences. It then delves into her journey, focusing on her education, career, and personal growth. The narrative also touches upon her relationships, challenges, and aspirations, providing a comprehensive view of her life.

The following text is a personal narrative for an individual based on their characteristics described below. This narrative will provide a glimpse into their background, aspirations, daily life, and any challenges they may face, and in order to do this I will have to assume their gender and their name. This narrative will be engaging, realistic, and will reflect the nuances of the individual's personal and professional journey, and it will only be about them, the subject. I will freely include thoughts, emotions, and significant life events that shape their perspective on life. 

-### Characteristics:
- Attended University: no
- Occupation Ca